<a href="https://colab.research.google.com/github/anuragN2107/SkyFlow-Aviation-BI-Platform/blob/main/SkyFlow_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Business Problem
Airlines operate in a highly volatile environment where operational disruptions directly damage brand equity and financial performance. While flight delays are a known operational reality, the exact threshold where an operational delay translates into a lost customer remains unclear.

Currently, the Operations department manages delays purely as an efficiency metric, while the Customer Experience (CX) team treats passenger dissatisfaction as a subjective survey problem. Because these datasets are kept in separate silos, the airline cannot answer a critical question: How do specific, compound operational failures (e.g., a weather delay combined with poor in-flight service) impact overall customer loyalty and satisfaction scores? Failing to quantify this relationship leads to misallocated recovery budgets and unmitigated customer churn.

# 2. Objective
The primary objective of this project is to build an end-to-end data pipeline and analytical model that bridges the gap between operational realities and customer sentiment.

* **Quantify the Friction Threshold:** Identify the exact duration and type of delay (Carrier, Weather, NAS, Security, Late Aircraft) that causes customer satisfaction ratings to drop sharply.

* **Feature Engineering across Silos:** Merge macroeconomic flight data (flights.csv) with granular customer survey metrics to find hidden correlations (e.g., Does a 30-minute delay matter less if the passenger is in Business Class vs. Economy?).

* **Predictive Risk Modeling:** Build a classification model to predict whether a passenger will report as "Satisfied" or "Neutral/Dissatisfied" based on the operational realities of their travel window.

# 3. Tools & Requirements
Technical Stack
**Language:** Python (v3.10+)

**Database Management System (DBMS):** Microsoft SQL Serve

**Data Processing:** pandas, NumPy

**Machine Learning / Analytics:** scikit-learn (or XGBoost / LightGBM)

**Data Visualization:** matplotlib, seaborn

**Visualization Layer:** Power Bi and Tableau

**Environment:** Google colab


In [2]:
import pandas as pd
import numpy as np

# 1. Define the missing column names in the exact order of your SQL query
column_headers = [
    'Flight_Date',
    'Airline_Code',
    'Flight_Num',
    'Origin_Airport',
    'Dest_Airport',
    'Scheduled_Dep_Time',
    'Distance',
    'Dep_Delay',
    'Is_Delayed'
]

# 2. Load the CSV
df = pd.read_csv('/content/flights_ml_sample.csv', header=None, names=column_headers)

#  Clean up the explicit text 'NULL' values that SQL Server exported
df = df.replace('NULL', np.nan)

#  Print verification to see your beautiful new clean dataframe
print(f"Dataset successfully loaded! Shape: {df.shape}")
df

Dataset successfully loaded! Shape: (500000, 9)


,Flight_Date,Airline_Code,Flight_Num,Origin_Airport,Dest_Airport,Scheduled_Dep_Time,Distance,Dep_Delay,Is_Delayed
0,2015-01-01,5,27,3,EV,5290,638.0,0,0
1,2015-01-01,1,3,6,B6,938,1353.0,0,0
2,2015-01-01,2,18,3,WN,1009,1641.0,0,0
3,2015-01-01,10,15,4,EV,2853,1904.0,14814,1
4,2015-01-01,5,13,3,WN,603,1809.0,0,0
...,...,...,...,...,...,...,...,...,...
499995,2015-01-01,4,22,3,WN,2167,2131.0,0,0
499996,2015-01-01,11,21,6,UA,571,1122.0,0,0
499997,2015-01-01,8,9,7,UA,1704,1619.0,0,0
499998,2015-01-01,12,13,7,WN,432,2158.0,0,0


In [3]:
df

,Flight_Date,Airline_Code,Flight_Num,Origin_Airport,Dest_Airport,Scheduled_Dep_Time,Distance,Dep_Delay,Is_Delayed
0,2015-01-01,5,27,3,EV,5290,638.0,0,0
1,2015-01-01,1,3,6,B6,938,1353.0,0,0
2,2015-01-01,2,18,3,WN,1009,1641.0,0,0
3,2015-01-01,10,15,4,EV,2853,1904.0,14814,1
4,2015-01-01,5,13,3,WN,603,1809.0,0,0
...,...,...,...,...,...,...,...,...,...
499995,2015-01-01,4,22,3,WN,2167,2131.0,0,0
499996,2015-01-01,11,21,6,UA,571,1122.0,0,0
499997,2015-01-01,8,9,7,UA,1704,1619.0,0,0
499998,2015-01-01,12,13,7,WN,432,2158.0,0,0


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [5]:
# Date Features
df['Flight_Date'] = pd.to_datetime(df['Flight_Date'])
df['Month'] = df['Flight_Date'].dt.month
df['DayOfWeek'] = df['Flight_Date'].dt.dayofweek

In [6]:
# Clean & convert numeric types
df['Scheduled_Dep_Time'] = pd.to_numeric(df['Scheduled_Dep_Time'], errors='coerce')
df['Distance'] = pd.to_numeric(df['Distance'], errors='coerce')
df['Is_Delayed'] = pd.to_numeric(df['Is_Delayed'], errors='coerce')

In [7]:
# Drop missing values
df = df.dropna()

In [8]:
# Extract hour element from timestamp
df['Dep_Hour'] = (df['Scheduled_Dep_Time'] / 100).astype(int)

/tmp/ipykernel_1302/2753718368.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Dep_Hour'] = (df['Scheduled_Dep_Time'] / 100).astype(int)


In [9]:
# Encode Categorical Fields
le_airline = LabelEncoder()
le_origin = LabelEncoder()
le_dest = LabelEncoder()

df['Airline_Encoded'] = le_airline.fit_transform(df['Airline_Code'].astype(str))
df['Origin_Encoded'] = le_origin.fit_transform(df['Origin_Airport'].astype(str))
df['Dest_Encoded'] = le_dest.fit_transform(df['Dest_Airport'].astype(str))

/tmp/ipykernel_1302/1771467815.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Airline_Encoded'] = le_airline.fit_transform(df['Airline_Code'].astype(str))
/tmp/ipykernel_1302/1771467815.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Origin_Encoded'] = le_origin.fit_transform(df['Origin_Airport'].astype(str))
/tmp/ipykernel_1302/1771467815.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

In [10]:
# Create final feature matrix
features = ['Month', 'DayOfWeek', 'Dep_Hour', 'Distance', 'Airline_Encoded', 'Origin_Encoded', 'Dest_Encoded']
X = df[features]
y = df['Is_Delayed'].astype(int)

print("Features mapped cleanly! Data ready for training.")

Features mapped cleanly! Data ready for training.


In [11]:
#Trainning the model

from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit LightGBM Model
model = LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42)
print("Training the model...")
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("\n=== Model Metrics ===")
print(classification_report(y_test, y_pred))

Training the model...
[LightGBM] [Info] Number of positive: 33189, number of negative: 360483
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007943 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 360
[LightGBM] [Info] Number of data points in the train set: 393672, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.084306 -> initscore=-2.385226
[LightGBM] [Info] Start training from score -2.385226
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

In [12]:
# NLP Sentiment Analyser
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [13]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [14]:
# Load train and test sets
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

# Combine them into a single dataset
reviews_df = pd.concat([train_df, test_df], ignore_index=True)
print(f"Data combined successfully! Shape: {reviews_df.shape}")

# Drop unneeded system index columns if they exist
if 'Unnamed: 0' in reviews_df.columns:
    reviews_df = reviews_df.drop(columns=['Unnamed: 0'])


Data combined successfully! Shape: (129880, 25)


In [15]:
# Generate Synthetic Customer feedback text
def generate_review_text(row):
    text = f"Flight in {row['Class']} class for a {row['Type of Travel']}. "
    text += f"Seat comfort rating: {row['Seat comfort']}/5. "
    text += f"Inflight entertainment: {row['Inflight entertainment']}/5. "
    text += f"Baggage handling: {row['Baggage handling']}/5. "

    if row['Arrival Delay in Minutes'] > 15:
        text += f"The flight was delayed by {int(row['Arrival Delay in Minutes'])} minutes which was frustrating. "
    else:
        text += "The arrival execution was smooth and timely. "

    if row['satisfaction'] == 'satisfied':
        text += "Overall, I am highly satisfied with this airline service."
    else:
        text += "Overall, I am dissatisfied and disappointed with the experience."
    return text

print("Synthesizing written narrative reviews from survey dimensions.")
reviews_df['Review_Text'] = reviews_df.apply(generate_review_text, axis=1)

Synthesizing written narrative reviews from survey dimensions.


In [16]:
# Run the Vader Sentiment Analyzer

sia = SentimentIntensityAnalyzer()
print("Analyzing text sentiment polarity metrics.")

# Extract the continuous compound score
reviews_df['Sentiment_Score'] = reviews_df['Review_Text'].apply(lambda x: sia.polarity_scores(x)['compound'])

# Categorize into clean visual attributes
def categorize_sentiment(score):
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

reviews_df['Sentiment_Category'] = reviews_df['Sentiment_Score'].apply(categorize_sentiment)


Analyzing text sentiment polarity metrics.


In [17]:
# Export Clean Analytical Layer
output_columns = [
    'id', 'Gender', 'Customer Type', 'Age', 'Type of Travel', 'Class',
    'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes',
    'satisfaction', 'Review_Text', 'Sentiment_Score', 'Sentiment_Category'
]
final_reviews_df = reviews_df[output_columns]

final_reviews_df.to_csv('enriched_customer_reviews.csv', index=False)
print("\n NLP pipeline complete! Saved 'Enriched_Customer_reviews.csv'")
print(final_reviews_df[['satisfaction', 'Sentiment_Score', 'Sentiment_Category']].head())


 NLP pipeline complete! Saved 'Enriched_Customer_reviews.csv'
              satisfaction  Sentiment_Score Sentiment_Category
0  neutral or dissatisfied          -0.6369           Negative
1  neutral or dissatisfied          -0.1027           Negative
2                satisfied           0.8122           Positive
3  neutral or dissatisfied          -0.1027           Negative
4                satisfied           0.8122           Positive
